In [3]:
import json, time, textwrap
import numpy as np
import requests

In [5]:
OLLAMA_URL = "http://localhost:11434"
LLM_MODEL = "llama3.1:8b"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
print("ready")

ready


In [7]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
print("device:", embedder.device)
print("dimensions:", embedder.get_sentence_embedding_dimension())

/home/jthe/Dev/tutes/geoscholar/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 14707.00it/s]


device: cuda:0
dimensions: 384


/tmp/ipykernel_16773/41192712.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("dimensions:", embedder.get_sentence_embedding_dimension())


In [8]:
sentences = [
    "SAR imagery can penetrate clouds and works at night.",
    "Radar satellites are able to see through cloud cover.",
    "NDVI is computed from red and near-infrared reflectance.",
    "Vegetation health is estimated from red and NIR bands.",
    "My favourite food is nasi goreng.",
]

emb = embedder.encode(sentences, normalize_embeddings=True)
print("shape:", emb.shape)

sims = emb @ emb.T
np.set_printoptions(precision=2, suppress=True)
print(sims)

shape: (5, 384)
[[1.   0.81 0.62 0.58 0.4 ]
 [0.81 1.   0.62 0.59 0.37]
 [0.62 0.62 1.   0.71 0.41]
 [0.58 0.59 0.71 1.   0.41]
 [0.4  0.37 0.41 0.41 1.  ]]


In [11]:
raw = embedder.encode(sentences[:2], normalize_embeddings=False)
a, b = raw[0], raw[1]
manual = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print(round(float(manual), 4), round(float(sims[0, 1]), 4))

0.809 0.809


In [21]:
r = requests.post(f"{OLLAMA_URL}/api/chat",
                  json={"model": LLM_MODEL, "messages": [{"role": "user", "content": "hi"}], "stream": False})
print(r.status_code, r.text)
print("LLM_MODEL is:", repr(LLM_MODEL))

404 {"error":"model 'llama3.1:8b' not found"}
LLM_MODEL is: 'llama3.1:8b'


In [17]:
# Messages: chat models take a list of messages with roles. systems sets rules; user is the question.
# Temperature: control randomness. Near 0 give focused, repeatable answers.
# Streaming: turn it off for now

def chat(prompt, system=None, temperature=0.1):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    resp = requests.post(
        f"{OLLAMA_URL}/api/chat", 
        json={
            "model": LLM_MODEL,
            "messages": messages,
            "stream": False,
            "options": {"temperature": temperature},
        },
        timeout=300,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"]

start = time.time()
print(chat("Explain the diferrence between optical and SAR satellite imagery in 3 bullet points."))
print(f"\n{time.time() - start:.1f} s")



Here are 3 bullet points explaining the difference between optical and SAR (Synthetic Aperture Radar) satellite imagery:

• **Source of Data**: Optical satellite imagery is captured using visible, infrared, or other forms of electromagnetic radiation, similar to how a camera works. SAR imagery, on the other hand, uses radar waves to capture data, which are emitted and received by the satellite.

• **Day/Night Capability**: Optical satellite imagery requires sunlight to capture images, so it's typically limited to daytime or twilight hours. SAR imagery, however, can capture images at any time of day or night, as it uses radar waves that can penetrate clouds and darkness.

• **Resolution and Data Quality**: Optical satellite imagery typically has higher spatial resolution (e.g., 1-5 meters) and can provide more detailed information about the scene, such as vegetation health, soil moisture, and atmospheric conditions. SAR imagery, while having lower spatial resolution (e.g., 10-100 meters

In [19]:
q = ("List three remote sensing papers published on arXiv in 2025 about "
     "flood mapping with Sentinel-1, with their authors and arXiv IDs.")
print(chat(q))

I'm unable to provide you with specific papers published on arXiv in 2025 about flood mapping with Sentinel-1. My current capabilities do not extend to predicting future publications or accessing information not yet available.


In [21]:
# Cell 7: fetch abstracts from the arXiv API
import feedparser

ARXIV_URL = "https://export.arxiv.org/api/query"
TOPICS = [
    "remote sensing",
    "synthetic aperture radar",
    "hyperspectral",
    "land cover",
    "flood mapping",
]

def clean(text):
    return " ".join(text.split())   # collapse newlines and double spaces

def fetch_topic(topic, max_results=10):
    query = f'abs:"{topic}" AND (cat:cs.CV OR cat:eess.IV)'
    resp = requests.get(
        ARXIV_URL,
        params={
            "search_query": query,
            "start": 0,
            "max_results": max_results,
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        },
        timeout=60,
    )
    resp.raise_for_status()
    feed = feedparser.parse(resp.text)
    papers = []
    for e in feed.entries:
        papers.append({
            "id": e.id.split("/abs/")[-1],          # e.g. 2410.01234v1
            "title": clean(e.title),
            "abstract": clean(e.summary),
            "authors": [a.name for a in e.authors],
            "published": e.published[:10],
            "url": e.id,
            "topic": topic,
        })
    return papers

corpus, seen = [], set()
for topic in TOPICS:
    for p in fetch_topic(topic):
        if p["id"] not in seen:              # the same paper can match two topics
            seen.add(p["id"])
            corpus.append(p)
    print(f"{topic}: total so far {len(corpus)}")
    time.sleep(3)                            # be polite to arXiv

print(corpus[0]["title"])
print(corpus[0]["abstract"][:300], "...")

remote sensing: total so far 10
synthetic aperture radar: total so far 19
hyperspectral: total so far 29
land cover: total so far 37
flood mapping: total so far 47
Data-Driven Vision-Language Models for Remote Sensing: A Survey
During the deep learning era, innovations in remote sensing (RS) vision models primarily focused on optimizing network architectures for specific tasks and conducting end-to-end training. However, with the advent of the Vision-Language Models (VLMs) era, constrained by substantial computational reso ...


In [23]:
# Cell 8: save the corpus so you don't re-download every time
import os
os.makedirs("data", exist_ok=True)
with open("data/abstracts.json", "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)
print("saved", len(corpus), "papers")

saved 47 papers


In [25]:
with open("data/abstracts.json", encoding="utf-8") as f:
    corpus = json.load(f)

In [27]:
# Cell 9: build the index
def doc_text(p):
    return f"{p['title']}. {p['abstract']}"

docs = [doc_text(p) for p in corpus]
doc_emb = embedder.encode(docs, normalize_embeddings=True, show_progress_bar=True)
print(doc_emb.shape)                      # (number_of_papers, 384)
np.save("data/doc_emb.npy", doc_emb)

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  6.53it/s]

(47, 384)


In [29]:
# Cell 10: brute-force cosine search
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def search(query, k=3, use_prefix=True):
    q = (QUERY_PREFIX + query) if use_prefix else query
    q_emb = embedder.encode(q, normalize_embeddings=True)
    scores = doc_emb @ q_emb                # one similarity score per paper
    top = np.argsort(-scores)[:k]           # indices of the k highest scores
    return [(float(scores[i]), corpus[i]) for i in top]

for score, p in search("detecting flooded areas with radar satellites"):
    print(f"{score:.3f}  [{p['id']}]  {p['title']}")

0.807  [2606.07780v1]  Land cover and flood type govern the detection limits of satellite-based flood mapping across diverse global flood events
0.801  [2608.02315v1]  GEOID-Flood: A Large-Scale Multi-Modal Benchmark Dataset for Flood Segmentation
0.795  [2606.29134v1]  Beyond Backscatter: AlphaEarth Land-Cover Priors for Rapid SAR Flood Segmentation Across Foundation Backbones


In [31]:
# Cell 11: try different kinds of queries
queries = [
    "deep learning for classifying land use from satellite images",
    "spectral bands beyond visible light for material identification",  # hyperspectral, without saying it
    "Sentinel-1",                                                          # an exact name
    "how to cook rice",                                                    # out of domain
]
for q in queries:
    print("\nQ:", q)
    for score, p in search(q):
        print(f"  {score:.3f}  {p['title'][:90]}")


Q: deep learning for classifying land use from satellite images
  0.854  Deep Learning for Forest Plantation Mapping in Godavari Districts of Andhra Pradesh, India
  0.816  Data-Driven Vision-Language Models for Remote Sensing: A Survey
  0.802  Evaluating the Alignment Between GeoAI Explanations and Domain Knowledge in Satellite-Base

Q: spectral bands beyond visible light for material identification
  0.762  Material and Defocus Cues for Occluded Object Localization in Single Hyperspectral Images:
  0.762  Spectral World Renderer: Unified 3D Hyperspectral Unmixing
  0.716  From Proper Scoring Rules and Edge Biases to Mumford-Shah Models: Robust Unsupervised Hype

Q: Sentinel-1
  0.684  Extending Dynamic World Surface Water Mapping to Sentinel-1 with AlphaEarth Embeddings
  0.632  High-Resolution Flood Mapping With Sentinel-1 and Sentinel-2 via Misalignment-Robust Cross
  0.632  GeoSET: Generalist Foundation Model for SAR-to-EO Image Translation

Q: how to cook rice
  0.535  SAR2Agri

In [33]:
# Cell 12: prompt construction
SYSTEM_PROMPT = """You are a research assistant for remote sensing literature.
Answer the question using ONLY the context below.
After each claim, cite the paper ID in square brackets, e.g. [2410.01234v1].
If the context does not contain the answer, reply exactly:
"I could not find this in the indexed papers."
Do not use outside knowledge. Be concise."""

def build_prompt(question, hits):
    blocks = []
    for score, p in hits:
        blocks.append(f"[{p['id']}] {p['title']} ({p['published']})\n{p['abstract']}")
    context = "\n\n---\n\n".join(blocks)
    return f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"

print(build_prompt("test question", search("flood mapping", k=2))[:800])

Context:
[2606.07780v1] Land cover and flood type govern the detection limits of satellite-based flood mapping across diverse global flood events (2026-06-05)
Floods are among the most destructive natural hazards, and their increasing frequency under climate change makes satellite-based inundation mapping essential for disaster response. Geospatial foundation models pretrained on satellite archives offer geographic transferability, but their operational reliability across diverse, unseen events remains uncharacterized. Here we deploy Prithvi-EO-2.0 across 19 out-of-distribution flood events (2017-2025) spanning six continents, eight climate zones, and six flood mechanisms, validating against two independent reference products. Detection accuracy depended jointly on land cover and flood typ


In [35]:
# Cell 13: naive RAG end to end
def ask(question, k=3, show_sources=True):
    t0 = time.time()
    hits = search(question, k=k)
    t1 = time.time()
    answer = chat(build_prompt(question, hits), system=SYSTEM_PROMPT)
    t2 = time.time()

    print(textwrap.fill(answer, width=100))
    if show_sources:
        print("\nSources:")
        for score, p in hits:
            print(f"  {score:.3f}  [{p['id']}] {p['title'][:80]}\n         {p['url']}")
    print(f"\nretrieval {t1 - t0:.2f} s | generation {t2 - t1:.1f} s")
    return answer, hits

answer, hits = ask("What methods do recent papers use for flood mapping with SAR imagery?")

The recent papers use the following methods for flood mapping with SAR imagery:  1. Cross-
Polarization Fusion of VV and VH SAR Observations [2605.02153v1] 2. Conditional Variational
Autoencoder (CVAE) for generative despeckling [2606.30511v1] 3. AlphaEarth Land-Cover Priors for
Rapid SAR Flood Segmentation [2606.29134v1] 4. Shift-invariant loss function to tolerate residual
geolocation uncertainty [2606.30511v1]  These methods aim to address the limitations of SAR imagery,
such as speckle noise and sensor co-registration uncertainty, to improve the accuracy of flood
mapping.

Sources:
  0.833  [2606.30511v1] High-Resolution Flood Mapping With Sentinel-1 and Sentinel-2 via Misalignment-Ro
         http://arxiv.org/abs/2606.30511v1
  0.818  [2606.29134v1] Beyond Backscatter: AlphaEarth Land-Cover Priors for Rapid SAR Flood Segmentatio
         http://arxiv.org/abs/2606.29134v1
  0.802  [2605.02153v1] Cross-Polarization Fusion of VV AND VH SAR Observations for Improved Flood Mappi
      

In [37]:
# Test with out of domain question
def ask(question, k=3, show_sources=True):
    t0 = time.time()
    hits = search(question, k=k)
    t1 = time.time()
    answer = chat(build_prompt(question, hits), system=SYSTEM_PROMPT)
    t2 = time.time()

    print(textwrap.fill(answer, width=100))
    if show_sources:
        print("\nSources:")
        for score, p in hits:
            print(f"  {score:.3f}  [{p['id']}] {p['title'][:80]}\n         {p['url']}")
    print(f"\nretrieval {t1 - t0:.2f} s | generation {t2 - t1:.1f} s")
    return answer, hits

answer, hits = ask("How do I cook Rendang?")

I could not find this in the indexed papers.

Sources:
  0.457  [2608.11142v1] SAR2Agri: Learning SAR Intensity Representations for Agricultural Monitoring
         http://arxiv.org/abs/2608.11142v1
  0.452  [2610.23609v1] ADA 2025: A Multimodal Dataset for Water-Deficit Stress Detection in Sugar Beets
         http://arxiv.org/abs/2610.23609v1
  0.449  [2610.04066v1] Evaluating Zone-Guided Front Extraction for Glacier Calving-Front Delineation in
         http://arxiv.org/abs/2610.04066v1

retrieval 0.02 s | generation 0.7 s
